'''
# 5. main_script.py ---
**This script runs cell classification on cellpose segmented images using a trained ResNet model.**
**!**The class index–to–label mapping used in the regionprop function must match the order used during model training.
Default mapping:
    0 → Early S
    1 → G1/G2
    2 → Late S
    3 → Mid S
    4 → Ambiguous
Please adjust this mapping if your model was trained with a different class order.
You can find the correct index order in your training script or in the saved idx_to_class dictionary.

structure:
-input_folder
-output_root
    -output_folder 
        -ori_img_folder (nd2, tif)
            -segmentation(mask.png/tif)
                -vis(png)
            -results(csv)
            -eightbit(tif)
            -opencv(tif)
        -channel_folder (png, json)
    -patches_root
        -phase_folders (5 classes)
'''

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import torch
from torchvision import models, transforms
from torchvision.transforms import v2
from skimage.io import imread
from skimage.measure import regionprops
from PIL import Image
import tifffile
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path

from model_loader import load_resnet_model
from img_utils import expand_region, convert_8bit_mask_background
from classification_utils import get_class_mapping, get_class_colors, create_result_table
from visualization_utils import save_expansion_visualizations

# from torchvision.transforms.v2 import Resize

In [ ]:
# --- Configuration ---
# base_dir = "/scratch/practical_students_bioimaging/yaziliu/project_cellcycle_classification_demo/lyz-2023dec13IF pcna esc gfp validate/ori_img_for_patches"
# base_name = "U1gfp 2203 004"


# input paths: base in_path plus subdirectory for tiff files (with pattern)
in_path = '/Users/david/Desktop/project_cellcycle_classification_demo_frcnn/test_edu_label_esc'

image_subdirectory = 'tif' # for resaved: "tif"
image_file_pattern = '*_ch1.tif' # e.g. "*_ch0.tif" to only include images of one channel

mask_subdirectory = 'segmentation_nuclei'
mask_file_patterns = '*.tif'

results_subdirectory = 'cellcycle_classification'
visualization_subdirectory = 'vis'

model_info_path = '/Users/david/Desktop/project_cellcycle_classification_demo_frcnn/resnet18_cellcycle.json'

In [ ]:
with open(model_info_path) as fd:
    model_info = json.load(fd)

weigths_path = Path(model_info_path).parent / model_info['weights_file']

# check model architecture
if model_info['architecture'] != 'resnet18':
    raise ValueError(f'Only architectures [\'resnet18\'] supported at the moment, was give {model_info['architecture']}')

# --- Load Model ---
model = models.resnet18(num_classes=len(model_info['classes']))
weights = torch.load(weigths_path, map_location='cpu')
model.load_state_dict(weights)

# TODO: make size settable
inference_transform = v2.Compose([
    v2.Resize((224, 224)),
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
])

In [ ]:

if torch.cuda.is_available():
    device = torch.device('cuda:0')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
    
model.to(device);

In [ ]:
image_files = sorted((Path(in_path) / image_subdirectory).glob(image_file_pattern))
mask_files = sorted((Path(in_path) / mask_subdirectory).glob(mask_file_patterns))

In [ ]:
img_path, mask_path = next(zip(image_files, mask_files))

In [ ]:
# # Define output subfolders 
# mask_dir, csv_dir, eight_bit_dir, cv2_dir = [
#     os.path.join(base_dir, name)
#     for name in ["segmentation", "results", "eightbit", "opencv"]
# ]

# for path in [mask_dir, csv_dir, eight_bit_dir, cv2_dir]:
#     os.makedirs(path, exist_ok=True)#instead of repeating os.path.join and os.makedirs

out_dir = Path(in_path) / results_subdirectory
visualization_dir = out_dir / visualization_subdirectory

if not out_dir.exists():
    out_dir.mkdir()
if not visualization_dir.exists():
    visualization_dir.mkdir()



In [ ]:
from skimage.morphology import dilation
from calmutils.morphology.structuring_elements import hypersphere_centered


for img_path, mask_path in zip(image_files, mask_files):

    # output file paths 
    annotated_img_path = visualization_dir / (img_path.stem + '.png')
    csv_path = out_dir / (img_path.stem + '_classification_results.csv')
    
    # --- Load and Expand Mask ---
    mask = imread(mask_path)
    img = imread(img_path)
    mask = dilation(mask, hypersphere_centered(mask.ndim, 3))
    
    
    # --- Classification ---
    idx_to_class = get_class_mapping()
    class_colors = get_class_colors()
    table_dict = create_result_table()
    
    # 8bit image for classification visualization
    img_8bit = convert_8bit_mask_background(img, mask)
    annotated_img = cv2.cvtColor(img_8bit, cv2.COLOR_GRAY2BGR)

    # TODO: get patches first, classify in batch
    
    for rprop in regionprops(mask, intensity_image=img):
        img_patch = img[rprop.slice]
        if img_patch.ndim == 2:
            img_patch = np.stack([img_patch]*3, axis=-1)
        # pil_patch = Image.fromarray(img_patch)
        input_tensor = inference_transform(img_patch).unsqueeze(0).to(device)
    
        # print(input_tensor)
    
        with torch.no_grad():
            logits = model(input_tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy().squeeze()
    
        class_idx = np.argmax(probs)
        class_name = idx_to_class[class_idx]
    
        table_dict['cell_id'].append(rprop.label)
        table_dict['prob_early'].append(probs[0])
        table_dict['prob_g1g2'].append(probs[1])
        table_dict['prob_late'].append(probs[2])
        table_dict['prob_mid'].append(probs[3])
        table_dict['prob_ambiguous'].append(probs[4]) 
        table_dict['max_class'].append(class_idx)
        table_dict['max_class_name'].append(class_name)
    
        if class_name != 'ambiguousXX':
            minr, minc, maxr, maxc = rprop.bbox
            color = class_colors[class_name]
            cv2.rectangle(annotated_img, (minc, minr), (maxc, maxr), color, 1)
            text_y = max(minr - 5, 10)
            cv2.putText(annotated_img, class_name, (minc, text_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)
    
    # --- Save Results ---
    cv2.imwrite(annotated_img_path, annotated_img)
    df = pd.DataFrame.from_dict(table_dict)
    df.to_csv(csv_path, index=False)
    
    print(f"Annotated image saved to: {annotated_img_path}")
    # print(f"Classification finished. Results saved to: {csv_path}")
    # print("\nFirst 5 rows of the result table:")
    # print(df.head())

In [ ]:
# Visualize annotated image + mask
img = imread(annotated_img_path)
mask = imread(mask_path)

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

axs[0].imshow(img)
axs[0].set_title("Annotated Classification Image")
axs[0].axis('off')

axs[1].imshow(mask, cmap='nipy_spectral')
axs[1].set_title("Segmentation Mask")
axs[1].axis('off')

plt.tight_layout()
plt.show()
